In [ ]:
import os
from pathlib import Path

os.chdir(Path.cwd().parent) #切換到根目錄

In [15]:
from pathlib import Path
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# 1) 路徑：依你的實際輸出位置調整
train_dir = Path("data/processed/splits/train")
val_dir   = Path("data/processed/splits/val")

print("train_dir exists:", train_dir.exists(), train_dir)
print("val_dir exists  :", val_dir.exists(), val_dir)

# 2) transforms：先用最基本可跑版本（Debug時越簡單越好）
img_size = 224
tfm = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
])

# 3) Dataset（ImageFolder 會自動把資料夾名當 label）
train_ds = datasets.ImageFolder(train_dir, transform=tfm)

print("num_classes:", len(train_ds.classes))
print("classes:", train_ds.classes)
print("class_to_idx:", train_ds.class_to_idx)
print("num_train_images:", len(train_ds))

# 4) DataLoader：Windows debug 先用 num_workers=0（最穩、最好抓錯）
batch_size = 32
train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
)

# 5) 取一個 batch，輸出 images.shape 與 labels
images, labels = next(iter(train_loader))

print("images.shape:", images.shape)   # 期待: torch.Size([32, 3, 224, 224])
print("labels.shape:", labels.shape)   # 期待: torch.Size([32])
print("labels (first 10):", labels[:10].tolist())

# (可選) 把 labels 轉回類別名稱，方便你肉眼確認
idx_to_class = {v: k for k, v in train_ds.class_to_idx.items()}
print("label names (first 10):", [idx_to_class[int(i)] for i in labels[:10]])


train_dir exists: True data\processed\splits\train
val_dir exists  : True data\processed\splits\val
num_classes: 6
classes: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
class_to_idx: {'buildings': 0, 'forest': 1, 'glacier': 2, 'mountain': 3, 'sea': 4, 'street': 5}
num_train_images: 9822
images.shape: torch.Size([32, 3, 224, 224])
labels.shape: torch.Size([32])
labels (first 10): [2, 5, 2, 4, 3, 2, 2, 2, 0, 5]
label names (first 10): ['glacier', 'street', 'glacier', 'sea', 'mountain', 'glacier', 'glacier', 'glacier', 'buildings', 'street']


In [16]:
# 驗證跑幾個 batch 不報錯（先跑 50 個就夠抓大多數問題）
max_batches = 50

try:
    for i, (x, y) in enumerate(train_loader):
        # 基本 sanity check（可快速抓 shape/label 異常）
        assert x.ndim == 4 and x.shape[1] == 3, f"bad x shape: {x.shape}"
        assert y.ndim == 1, f"bad y shape: {y.shape}"
        if i == 0:
            print("first batch ok:", x.shape, y.shape)
        if i + 1 >= max_batches:
            break

    print(f"DataLoader OK: iterated {min(max_batches, i+1)} batches without error.")

except Exception as e:
    print("DataLoader ERROR:", type(e).__name__, str(e))
    raise


first batch ok: torch.Size([32, 3, 224, 224]) torch.Size([32])
DataLoader OK: iterated 50 batches without error.
